In [1]:
import pandas as pd

df = pd.read_csv("messy_sales.csv")

print("Dataset shape:", df.shape)

display(df.head(10))

print("\nColumn types:")
print(df.dtypes)

Dataset shape: (300, 6)


,order_id,date,product,price,qty,zip
0,1254,04/06/2026,mug,11.36,4,60614
1,1057,2026-05-30,charger,14.79,3,30303
2,1150,05/06/2026,mug,5.50,1,10001
3,1066,05/30/2026,notebook,NaN,1,98101
4,1114,04/06/2026,webcam,45.59,4,90405
5,1280,04/22/2026,pen set,NaN,4,30303
6,1204,04/18/2026,charger,8.41,2,2134
7,1249,2026-05-15,keyboard,75.04,4,90405
8,1037,2026-05-19,mug,9.31,2,60614
9,1177,2026-05-13,mug,10.11,3,90210



Column types:
order_id      int64
date            str
product         str
price       float64
qty           int64
zip           int64
dtype: object


## Initial Observations

1. The price column contains missing values that need to be investigated and handled.
2. The zip column is stored as a number, which can cause leading zeros to be lost.
3. The date column contains mixed date formats, and the quantity column includes negative values that need investigation.

In [2]:
df.isna().sum()

order_id     0
date         0
product      0
price       12
qty          0
zip          0
dtype: int64

In [3]:
fill_value = df["price"].median()

print("Median price used:", fill_value)

df["price"] = df["price"].fillna(fill_value)

print("\nMissing values after filling:")
print(df.isna().sum())

Median price used: 37.53

Missing values after filling:
order_id    0
date        0
product     0
price       0
qty         0
zip         0
dtype: int64


In [4]:
print("Shape before removing duplicates:", df.shape)

duplicate_count = df.duplicated().sum()
print("Duplicate rows found:", duplicate_count)

df = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)

Shape before removing duplicates: (300, 6)
Duplicate rows found: 8
Shape after removing duplicates: (292, 6)


In [5]:
df["zip"] = df["zip"].astype(str).str.zfill(5)

print(df["zip"].head(10))

print("\nZIP code lengths:")
print(df["zip"].str.len().value_counts())

0    60614
1    30303
2    10001
3    98101
4    90405
5    30303
6    02134
7    90405
8    60614
9    90210
Name: zip, dtype: str

ZIP code lengths:
zip
5    292
Name: count, dtype: int64


In [6]:
df["date"] = pd.to_datetime(df["date"], format="mixed")

print(df["date"].head(10))

print("\nColumn types after conversion:")
print(df.dtypes)

0   2026-04-06
1   2026-05-30
2   2026-05-06
3   2026-05-30
4   2026-04-06
5   2026-04-22
6   2026-04-18
7   2026-05-15
8   2026-05-19
9   2026-05-13
Name: date, dtype: datetime64[us]

Column types after conversion:
order_id             int64
date        datetime64[us]
product                str
price              float64
qty                  int64
zip                    str
dtype: object


In [7]:
negative_rows = df[df["qty"] < 0]

print("Number of rows with negative quantities:", len(negative_rows))

display(negative_rows)

Number of rows with negative quantities: 3


,order_id,date,product,price,qty,zip
202,1140,2026-04-18,webcam,38.69,-5,98101
262,1233,2026-05-09,desk lamp,22.64,-4,10001
297,1025,2026-04-27,keyboard,47.30,-2,02116


In [12]:
print("Final shape:", df.shape)

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate rows remaining:", df.duplicated().sum())

print("\nColumn types:")
print(df.dtypes)

print("\nZIP code lengths:")
print(df["zip"].str.len().value_counts())

print("\nNegative quantities retained:", (df["qty"] < 0).sum())

Final shape: (292, 6)

Missing values:
order_id    0
date        0
product     0
price       0
qty         0
zip         0
dtype: int64

Duplicate rows remaining: 0

Column types:
order_id             int64
date        datetime64[us]
product                str
price              float64
qty                  int64
zip                    str
dtype: object

ZIP code lengths:
zip
5    292
Name: count, dtype: int64

Negative quantities retained: 3


In [13]:
df.to_csv("sales_clean.csv", index=False)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.


In [14]:
cleaned_check = pd.read_csv("sales_clean.csv")

print("Saved file shape:", cleaned_check.shape)
display(cleaned_check.head())

Saved file shape: (292, 6)


,order_id,date,product,price,qty,zip
0,1254,2026-04-06,mug,11.36,4,60614
1,1057,2026-05-30,charger,14.79,3,30303
2,1150,2026-05-06,mug,5.50,1,10001
3,1066,2026-05-30,notebook,37.53,1,98101
4,1114,2026-04-06,webcam,45.59,4,90405


Cleaning Log
1. Initial Dataset

Loaded messy_sales.csv into pandas. The original dataset contained 300 rows and 6 columns.

2. Missing Prices

Identified 12 missing values in the price column. Filled these missing values using the median price of $37.53. The median was selected because it is less sensitive to extreme prices than the mean. After imputation, no missing values remained.

3. Duplicate Rows

Identified and removed 8 fully duplicated rows using drop_duplicates(). The dataset decreased from 300 rows to 292 rows, with 6 columns retained.

4. ZIP Code Formatting

Converted the zip column from numeric values to strings and used zfill(5) to restore five-character ZIP codes. This preserves leading zeros in ZIP codes.

5. Date Standardization

Converted the date column from mixed-format text to pandas datetime values using pd.to_datetime() with format="mixed". The resulting data type is datetime64[ns].

6. Negative Quantities

Identified 3 rows with negative quantities. Retained these records because they may represent product returns rather than data-entry errors. Without additional transaction information, deleting or correcting them would be an unsupported assumption.

7. Final Dataset

The cleaned dataset contains 292 rows and 6 columns. All missing prices have been filled, duplicate rows have been removed, ZIP codes have been standardized to five-character text values, and dates have been converted to datetime values. The cleaned data was exported to sales_clean.csv without an index column.

8. Potential Business Impact

An incorrect cleaning decision, such as removing legitimate returns or filling missing prices with an inappropriate value, could distort revenue calculations and lead a business to make inaccurate inventory, pricing, or sales decisions.